## Concept focus — Composing iterators with itertools

The `itertools` module gives you production-grade building blocks for sequence composition, grouping, slicing, and accumulation. Learning these patterns saves you from rewriting subtle iteration logic poorly by hand.

```text
input streams -> itertools helpers -> transformed stream

chain      : joins streams
islice     : takes a window
accumulate : running totals
groupby    : consecutive grouping
```

### How to think about it
Think in terms of transformations, not loops first. Ask: am I chaining streams, slicing them, grouping them, or computing rolling state? Once you name the operation, `itertools` often already has the correct primitive.

### Visual references and further study
- [itertools documentation](https://docs.python.org/3/library/itertools.html)
- [Python itertools recipes](https://docs.python.org/3/library/itertools.html#itertools-recipes)
- [More Itertools](https://more-itertools.readthedocs.io/)
- [Python Tutor visualizer](https://pythontutor.com/visualize.html)

---

# Module 14 — Iterators, Generators, and Lazy Pipelines

## Exercise 14.3 — Twenty problems with one-line answers

Part A: solve each with itertools (or a builtin). One line each.
Part B: implement six of them yourself, lazily, without itertools.
Run:  python ex03_itertools.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. The iterator protocol

Two methods, and everything else in this module is built on them.

In [ ]:
iter(obj)      # -> obj.__iter__()  : returns an ITERATOR
next(it)       # -> it.__next__()   : the next value, or raises StopIteration

A `for` loop is sugar for exactly this:

In [ ]:
for x in things: process(x)

# is precisely:
it = iter(things)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    process(x)

| | Iterable | Iterator |
|---|---|---|
| Defines | `__iter__` | `__iter__` **and** `__next__` |
| `__iter__` returns | a **fresh** iterator | `self` |
| Reusable | yes | **no** |
| Examples | `list`, `dict`, `str`, `range` | generators, file objects, `iter([])` |

**Every iterator is an iterable; not every iterable is an iterator.** The
distinction shows up as the one-shot bug (Module 09), and it is worth being able
to state precisely:

In [ ]:
data = (x for x in range(3))
list(data)      # [0, 1, 2]
list(data)      # []          <- exhausted, silently

No error. That silence is the whole hazard.

---

## Concept 2. Generator functions

Any function containing `yield` is a generator function. Calling it **runs
nothing** — it returns a generator object.

In [ ]:
def countdown(n: int):
    print("starting")            # does NOT run on the call
    while n > 0:
        yield n
        n -= 1
    print("done")

gen = countdown(3)               # nothing printed
next(gen)                         # 'starting', then 3
next(gen)                         # 2

`yield` **suspends** the function: locals, instruction pointer, and the whole
frame are preserved. `next()` resumes exactly where it stopped. That suspended
frame is the mental image to carry — it is also how `await` works (Module 22).

### Generators are the easiest way to write `__iter__`

Compare this with Module 09's iterator class:

In [ ]:
class Countdown:
    def __init__(self, start: int) -> None:
        self.start = start

    def __iter__(self):
        current = self.start      # a LOCAL, so each call gets fresh state
        while current > 0:
            yield current
            current -= 1

Two `for` loops both work, because each call to `__iter__` creates a new
generator with its own locals. That is the fix for the one-shot bug, and it is
free.

### `yield from`

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)      # delegate, recursively
        else:
            yield item

`yield from x` is not just `for i in x: yield i` — it also forwards `send`,
`throw` and `close`, and propagates the sub-generator's return value. For plain
iteration the loop is equivalent; for coroutines it is not.

### Generator expressions

In [ ]:
squares = (x * x for x in range(1_000_000))     # lazy, ~200 bytes
squares = [x * x for x in range(1_000_000)]     # eager, ~40 MB

sum(x * x for x in data)                         # parens optional as sole arg
any(line.startswith("ERROR") for line in fh)     # short-circuits

**Use a generator expression when the values are consumed once.** Use a list
when you need to index, re-iterate, or take `len()`.

---

## Concept 3. Pipelines

The technique that makes this module worth its time. Each stage is lazy; the
data flows through one item at a time.

In [ ]:
def read_lines(path):
    with open(path, encoding="utf-8") as fh:
        yield from fh

def parse(lines):
    for line in lines:
        parts = line.rstrip("\n").split("\t")
        if len(parts) == 4:
            yield {"ts": parts[0], "level": parts[1],
                   "user": parts[2], "msg": parts[3]}

def only(records, level):
    for r in records:
        if r["level"] == level:
            yield r

def summarise(records, limit):
    for r in islice(records, limit):
        yield f"{r['ts']} {r['user']}: {r['msg']}"

# nothing has run yet
pipeline = summarise(only(parse(read_lines("50gb.log")), "ERROR"), 10)

for line in pipeline:      # NOW it runs, one line at a time
    print(line)

Memory: one line. Work done: it stops after finding ten errors, even if the file
is 50 GB and the tenth error is on line 900.

**Three properties that fall out:**

1. **Constant memory**, regardless of input size.
2. **Early termination** — `break` at any point stops all upstream work.
3. **Composability** — any stage can be inserted, removed, or reordered without
   touching the others.

### The `with` trap in a generator

In [ ]:
def read_lines(path):
    with open(path) as fh:
        yield from fh          # the file stays open while the generator lives

If the consumer abandons the generator, the `with` block exits when the
generator is garbage collected — which is *usually* immediate under CPython
refcounting and *not guaranteed* (Module 02). For long-lived programs, close it
explicitly or use `contextlib.closing`. This is a real source of "too many open
files" in production.

---

## Concept 4. `itertools`

The composable toolkit. Everything here is lazy.

In [ ]:
from itertools import (
    chain, islice, tee, cycle, repeat, count,
    groupby, takewhile, dropwhile, filterfalse, compress,
    accumulate, pairwise, product, permutations, combinations, zip_longest,
)

chain(a, b, c)                  # concatenate iterables
chain.from_iterable(nested)     # flatten one level -- the O(n) way
islice(it, 10)                  # a slice of an iterator
islice(it, 5, 15)
takewhile(lambda x: x < 100, it)   # stop at the first failure
dropwhile(lambda x: x < 100, it)   # skip until the first success
accumulate(nums)                    # running totals
pairwise("abcd")                    # ('a','b'), ('b','c'), ('c','d')  3.10+
groupby(sorted(rows, key=f), key=f) # group CONSECUTIVE equal keys
zip_longest(a, b, fillvalue=0)
count(1)                            # 1, 2, 3, ... infinite

**`groupby` requires sorted input.** It groups *consecutive* equal keys, like
Unix `uniq`. Unsorted input silently produces many small groups instead of one
per key — a quiet wrong answer, not an error. If you cannot sort (it is an
infinite stream, or sorting is too expensive), use `defaultdict(list)` instead.

**`tee` is not free.** `tee(it, 2)` buffers everything one branch has consumed
and the other has not. If one branch runs ahead, the buffer grows to that gap.
Two independent passes over a list are usually cheaper.

**Flattening:**

In [ ]:
sum(lists, [])                          # O(n^2). Never.
list(chain.from_iterable(lists))        # O(n). Always.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: The iterator protocol
- Section 2: Generator functions
- Section 3: Pipelines
- Section 4: `itertools`
- Section 5: Generators as coroutines
- Section 6: When *not* to be lazy

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from collections.abc import Iterable, Iterator
from typing import Any, TypeVar

T = TypeVar("T")


# --- Part A: one line each ----------------------------------------------------

---

## `p01_flatten`

[[1,2],[3],[4,5]] -> [1,2,3,4,5]. NOT sum(nested, []).

In [ ]:
def p01_flatten(nested: Iterable[Iterable[T]]) -> list[T]:
    """[[1,2],[3],[4,5]] -> [1,2,3,4,5]. NOT sum(nested, [])."""
    raise NotImplementedError

---

## `p02_first_n`

First n items of a possibly-infinite iterable.

In [ ]:
def p02_first_n(it: Iterable[T], n: int) -> list[T]:
    """First n items of a possibly-infinite iterable."""
    raise NotImplementedError

---

## `p03_chunks`

[1,2,3,4,5], 2 -> (1,2), (3,4), (5,). Lazily.

In [ ]:
def p03_chunks(it: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    """[1,2,3,4,5], 2 -> (1,2), (3,4), (5,). Lazily.
    3.12 has itertools.batched. Write it for 3.10 too."""
    raise NotImplementedError

---

## `p04_running_total`

[1,2,3] -> [1,3,6]

In [ ]:
def p04_running_total(nums: Iterable[float]) -> list[float]:
    """[1,2,3] -> [1,3,6]"""
    raise NotImplementedError

---

## `p05_consecutive_pairs`

'abcd' -> [('a','b'),('b','c'),('c','d')]

In [ ]:
def p05_consecutive_pairs(it: Iterable[T]) -> list[tuple[T, T]]:
    """'abcd' -> [('a','b'),('b','c'),('c','d')]"""
    raise NotImplementedError

---

## `p06_until_negative`

[1,2,-1,3] -> [1,2]  (stop at the first failure, do not filter)

In [ ]:
def p06_until_negative(nums: Iterable[float]) -> list[float]:
    """[1,2,-1,3] -> [1,2]  (stop at the first failure, do not filter)"""
    raise NotImplementedError

---

## `p07_skip_header`

Drop leading lines starting with '#', keep everything after -- including

In [ ]:
def p07_skip_header(lines: Iterable[str]) -> Iterator[str]:
    """Drop leading lines starting with '#', keep everything after -- including
    later '#' lines."""
    raise NotImplementedError

---

## `p08_group_by_key`

Group rows by a key. Careful: groupby needs sorted input. Show BOTH the

In [ ]:
def p08_group_by_key(rows: list[dict[str, Any]], key: str) -> dict[Any, list[dict]]:
    """Group rows by a key. Careful: groupby needs sorted input. Show BOTH the
    groupby version and the defaultdict version, and say which you would ship
    and why."""
    raise NotImplementedError

---

## `p09_round_robin`

'abc', 'de' -> a, d, b, e, c

In [ ]:
def p09_round_robin(*its: Iterable[T]) -> Iterator[T]:
    """'abc', 'de' -> a, d, b, e, c"""
    raise NotImplementedError

---

## `p10_all_pairs`

Every unordered pair, no repeats. [1,2,3] -> (1,2),(1,3),(2,3)

In [ ]:
def p10_all_pairs(items: Iterable[T]) -> list[tuple[T, T]]:
    """Every unordered pair, no repeats. [1,2,3] -> (1,2),(1,3),(2,3)"""
    raise NotImplementedError

---

## `p11_cartesian`

Every combination from two iterables.

In [ ]:
def p11_cartesian(a: Iterable[T], b: Iterable[T]) -> list[tuple[T, T]]:
    """Every combination from two iterables."""
    raise NotImplementedError

---

## `p12_cycle_n`

Repeat the sequence until `total` items have been produced.

In [ ]:
def p12_cycle_n(items: Iterable[T], total: int) -> list[T]:
    """Repeat the sequence until `total` items have been produced."""
    raise NotImplementedError

---

## `p13_zip_padded`

Zip without truncating the longer one.

In [ ]:
def p13_zip_padded(a: Iterable[T], b: Iterable[T], fill: Any) -> list[tuple]:
    """Zip without truncating the longer one."""
    raise NotImplementedError

---

## `p14_dedupe_consecutive`

[1,1,2,2,2,1] -> 1,2,1  (Unix uniq, NOT set())

In [ ]:
def p14_dedupe_consecutive(it: Iterable[T]) -> Iterator[T]:
    """[1,1,2,2,2,1] -> 1,2,1  (Unix uniq, NOT set())"""
    raise NotImplementedError

---

## `p15_select`

Keep items whose corresponding flag is True.

In [ ]:
def p15_select(items: Iterable[T], flags: Iterable[bool]) -> list[T]:
    """Keep items whose corresponding flag is True."""
    raise NotImplementedError

---

## `p16_split_at`

First n as a list, the REST as a lazy iterator. Careful: consuming the

In [ ]:
def p16_split_at(it: Iterable[T], n: int) -> tuple[list[T], Iterator[T]]:
    """First n as a list, the REST as a lazy iterator. Careful: consuming the
    first part must not consume the rest."""
    raise NotImplementedError

---

## `p17_nth`

The nth item of an iterator, without materialising it.

In [ ]:
def p17_nth(it: Iterable[T], n: int, default: Any = None) -> Any:
    """The nth item of an iterator, without materialising it."""
    raise NotImplementedError

---

## `p18_last`

The final item of an iterator, in constant memory.

In [ ]:
def p18_last(it: Iterable[T]) -> T:
    """The final item of an iterator, in constant memory."""
    raise NotImplementedError

---

## `p19_count_items`

len() for an iterator. Note what this costs.

In [ ]:
def p19_count_items(it: Iterable[Any]) -> int:
    """len() for an iterator. Note what this costs."""
    raise NotImplementedError

---

## `p20_interleave_longest`

Round-robin, but continue with the longer iterables after the short ones

In [ ]:
def p20_interleave_longest(*its: Iterable[T]) -> list[T]:
    """Round-robin, but continue with the longer iterables after the short ones
    are exhausted."""
    raise NotImplementedError

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    assert p01_flatten([[1, 2], [3], [4, 5]]) == [1, 2, 3, 4, 5]
    assert p02_first_n(iter(range(1000)), 3) == [0, 1, 2]
    assert list(p03_chunks([1, 2, 3, 4, 5], 2)) == [(1, 2), (3, 4), (5,)]
    assert p04_running_total([1, 2, 3]) == [1, 3, 6]
    assert p05_consecutive_pairs("abcd") == [("a", "b"), ("b", "c"), ("c", "d")]
    assert p06_until_negative([1, 2, -1, 3]) == [1, 2]
    assert list(p07_skip_header(["#a", "#b", "x", "#c", "y"])) == ["x", "#c", "y"]

    rows = [{"k": "a", "v": 1}, {"k": "b", "v": 2}, {"k": "a", "v": 3}]
    assert len(p08_group_by_key(rows, "k")["a"]) == 2

    assert list(p09_round_robin("abc", "de")) == ["a", "d", "b", "e", "c"]
    assert p10_all_pairs([1, 2, 3]) == [(1, 2), (1, 3), (2, 3)]
    assert len(p11_cartesian([1, 2], "ab")) == 4
    assert p12_cycle_n([1, 2], 5) == [1, 2, 1, 2, 1]
    assert p13_zip_padded([1, 2, 3], "ab", None) == [(1, "a"), (2, "b"), (3, None)]
    assert list(p14_dedupe_consecutive([1, 1, 2, 2, 2, 1])) == [1, 2, 1]
    assert p15_select("abcd", [1, 0, 1, 0]) == ["a", "c"]

    head, tail = p16_split_at(iter(range(10)), 3)
    assert head == [0, 1, 2] and list(tail) == list(range(3, 10))

    assert p17_nth(iter(range(10)), 5) == 5
    assert p17_nth(iter(range(3)), 99, "none") == "none"
    assert p18_last(iter(range(10))) == 9
    assert p19_count_items(iter(range(42))) == 42
    assert p20_interleave_longest("abc", "de") == ["a", "d", "b", "e", "c"]

    print("all 20 checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.